# s²ISM vs s²ISM-RbN

This notebook provides an example of **s²ISM reconstruction with RbN regularization**.

It compares:

- **s²ISM**: the conventional maximum-likelihood reconstruction of the detector-resolved acquisition.
- **s²ISM-RbN**: the same forward model applied jointly to several independent photon realizations, with a consistency regularization across them.

The independent realizations are generated with **binomial photon splitting**.

The notebook shows two ways to run s²ISM-RbN:

1. automatic selection of the regularization parameter $\lambda$ with the **Poisson Residual Whiteness Principle (PRWP)**;
2. reconstruction with a **fixed value of $\lambda$**.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import brighteyes.imaging.io.mcs as mcs
import brighteyes.imaging.analysis.Graph_lib as gra
import brighteyes.imaging.simulation.PSF_sim as psf_sim

import brighteyes.imaging.s2ism.psf_estimator as psf_est
from brighteyes.imaging.s2ism import s2ism as s2

from brighteyes.imaging.varreg import preprocessing as pp
from brighteyes.imaging.varreg import utils
from brighteyes.imaging.varreg import whiteness_principle as wp
from brighteyes.imaging.varreg import varreg

## 2. Parameters Settings

The optical values below are examples and should match the acquisition used in the notebook. If you use a different dataset, update the file path, wavelengths, and numerical aperture.

`N_REALIZATIONS` controls how many independent temporal photon realizations are built from the raw acquisition. The example uses **8**.

In [ ]:
# ---- Data ----
wd_original = Path.cwd()
parts = list(wd_original.parts)
idx = next((i for i, p in enumerate(parts) if p.lower() == "notebooks"), None)
wd = Path(*parts[:idx]) if idx is not None and idx > 0 else wd_original

DATA_PATH = (
    wd
    / "Data"
    / "Phalloidin_AF647_HeLa_PFA4_G0_2"
    / "Less_photons"
    / "06_Phalloidin_AF647-23-06-2026-17-45-08.h5"
)

# ---- Microscope parameters ----
NA = 1.49
EXCITATION_NM = 639
EMISSION_NM = 671

# ---- s²ISM model ----
N_AXIAL_PLANES = 2
IN_FOCUS_PLANE = 1

# ---- Reference s²ISM ----
S2ISM_MAX_ITER = 100

# ---- RbN parameters ----
N_REALIZATIONS = 8
LAMBDA_GRID = np.linspace(1, 5, 5)
FIXED_LAMBDA = 5
RBN_MAX_ITER = 2000

# ---- PRWP ----
PRWP_BORDER = 10

# Use CUDA when available. The varreg optimizer falls back to CPU automatically.
PROCESS = "gpu"

## 3. Load the BrightEyes-MCS acquisition

The expected raw layout is:

`height × width × time bins × detector channels`

For s²ISM we keep the detector channels separate, because each detector element has its own image-formation model.

In [ ]:
raw_data, metadata = mcs.load(str(DATA_PATH))
raw_data = np.asarray(raw_data).squeeze()

pixel_size_nm = metadata.dx * 1e3

print(f"Raw data shape: {raw_data.shape}")
print(f"Pixel size: {pixel_size_nm:.2f} nm")

## 4. Build the detector-resolved image

The time bins are summed to obtain one photon-count image for each detector channel.

For visualization only, we also sum the detector channels to obtain the corresponding confocal image.

In [ ]:
# Sum the temporal bins, while keeping the detector channels.
detector_image = raw_data.sum(axis=-2)       # (height, width, channels)

# Sum detector channels only for visualization.
confocal_image = detector_image.sum(axis=-1) # (height, width)

fig, ax = plt.subplots(figsize=(5, 5))
gra.ShowImg(confocal_image, pixel_size_nm * 1e-3, clabel="Counts", fig=fig, ax=ax)
ax.set_title("Raw confocal image")
plt.show()

## 5. Estimate the s²ISM PSFs

s²ISM uses a different PSF for each detector channel and axial plane.

The PSFs are calibrated from the detector-resolved acquisition using the `s2ism` tools.  
In this example the model contains one out-of-focus plane and one in-focus plane.

In [ ]:
excitation = psf_sim.simSettings()
excitation.na = NA
excitation.wl = EXCITATION_NM
excitation.mask_sampl = 101

emission = excitation.copy()
emission.wl = EMISSION_NM

grid = psf_sim.GridParameters()
grid.pxsizex = pixel_size_nm
grid.Nz = N_AXIAL_PLANES

# Estimate the distance of the out-of-focus plane.
z_out_of_focus, _ = psf_est.find_out_of_focus_from_param(
    grid.pxsizex,
    excitation,
    emission,
    mode="Pearson",
    stack="positive",
    graph=False,
)
grid.pxsizez = z_out_of_focus

# Detector- and plane-dependent PSFs: (Nz, Mx, My, channels).
psf_ism, _, _ = psf_est.psf_estimator_from_data(
    detector_image,
    excitation,
    emission,
    grid,
    z_out_of_focus=z_out_of_focus,
)

print(f"s²ISM PSF shape: {psf_ism.shape}")
print(f"Out-of-focus distance: {z_out_of_focus:.1f} nm")

# Display the detector PSFs of the in-focus plane.
fig = gra.ShowDataset(
    psf_ism[IN_FOCUS_PLANE],
    normalize=True,
    colorbar=True,
)
fig.suptitle("In-focus detector PSFs", y=1.02)
plt.show()

## 6. Create independent photon realizations

s²ISM-RbN needs several statistically independent measurements of the same object.

Here the full detector-resolved photon counts are split computationally into `K` realizations with `pp.binomial_split`.

The detector channels are **not summed**: each realization keeps the complete detector information required by the s²ISM forward model.

In [ ]:
# Binomial photon splitting:
# (height, width, channels) -> (height, width, channels, K)
detector_realizations = pp.binomial_split(
    detector_image,
    n_splits=N_REALIZATIONS,
).cpu().numpy()

print(f"s²ISM-RbN dataset shape: {detector_realizations.shape}")

# Show the confocal projection of a few realizations.
realization_preview = detector_realizations.sum(axis=-2)
n_show = min(4, N_REALIZATIONS)

fig = gra.ShowDataset(
    realization_preview[..., :n_show],
    normalize=True,
    colorbar=True,
)
fig.suptitle("Independent photon realizations", y=1.02)
plt.show()

## 7. Optional spatial weight

RbN can use a spatial weight so that the consistency penalty adapts to the local signal-to-noise ratio.

The weight is calculated from the channel-summed realizations and the same 2D map is used for every reconstructed object.

To use a uniform weight everywhere, simply replace `weight_map` with `1`.

In [ ]:
WEIGHT_SMOOTH_SIGMA = 2

channel_summed_realizations = detector_realizations.sum(axis=-2)

weight_map = utils.calculate_weight_map(
    channel_summed_realizations,
    smooth_sigma=WEIGHT_SMOOTH_SIGMA,
    normalize=True,
)[..., 0]

fig, ax = plt.subplots(figsize=(5, 5))
gra.ShowImg(weight_map, pixel_size_nm * 1e-3, fig=fig, ax=ax)
ax.set_title("RbN weight map")
plt.show()

## 8. Conventional s²ISM reconstruction

This is the reference reconstruction obtained from the full detector-resolved image.

Only the final reconstruction is saved. The displayed image is the reconstructed **in-focus plane**.

In [ ]:
s2ism_result, *_ = s2.max_likelihood_reconstruction(
    detector_image,
    psf_ism,
    max_iter=S2ISM_MAX_ITER,
    rep_to_save="last",
)

s2ism_volume = np.asarray(s2ism_result).squeeze()
s2ism_image = s2ism_volume[IN_FOCUS_PLANE]

fig, ax = plt.subplots(figsize=(5, 5))
gra.ShowImg(s2ism_image, pixel_size_nm * 1e-3, clabel="Counts", fig=fig, ax=ax)
ax.set_title(f"s²ISM ({S2ISM_MAX_ITER} iterations)")
plt.show()

## 9. s²ISM-RbN with automatic $\lambda$ selection

For every candidate $\lambda$, s²ISM-RbN jointly reconstructs the `K` photon realizations using the complete detector- and plane-dependent s²ISM forward model.

PRWP evaluates the residual of each reconstruction and selects the candidate with the lowest whiteness value.

Only the last reconstruction for each lambda is kept, avoiding storage of the complete iteration history.

In [ ]:
rbn_solver = wp.make_variance_reconstructor(
    dset=detector_realizations,
    psf=psf_ism,
    weight_map=weight_map,
    stop_tol=1e-6,
    stop_criteria="step_size",
    max_iter=RBN_MAX_ITER,
    method="sgp",
    regularizer_norm="l2",
    initialization="flat",
    normalize=True,
    process=PROCESS,
    idx_in_focus=IN_FOCUS_PLANE,
    rec_to_save="last",
    step_size_search="line_search",
    line_search_params=(1e18, 2),
    verbose=False,
)

prwp_device = torch.device(
    "cuda" if PROCESS == "gpu" and torch.cuda.is_available() else "cpu"
)

best_lambda, best_obj, best_whiteness = wp.choose_lambda_by_WP(
    solver_fn=rbn_solver,
    lambdas=LAMBDA_GRID,
    dset=torch.as_tensor(detector_realizations, dtype=torch.float32),
    psf=torch.as_tensor(psf_ism, dtype=torch.float32),
    masked=True,
    remove_border=PRWP_BORDER,
    idx_in_focus=IN_FOCUS_PLANE,
    device=prwp_device,
    verbose=True,
)

# Sum the K reconstruction-specific objects to recover the full photon flux.
rbn_volume = best_obj.sum(dim=-1).numpy()
rbn_image = rbn_volume[IN_FOCUS_PLANE]

print(f"Selected lambda: {best_lambda:.4g}")
print(f"Residual whiteness: {best_whiteness:.6g}")

## 9b. s²ISM-RbN with fixed $\lambda$

Use this cell when a value of $\lambda$ is already known and no PRWP search is needed.

`FIXED_LAMBDA` is defined in the settings cell at the top of the notebook.

In [ ]:
fixed_history, _ = varreg.variance_reconstruction(
    detector_realizations,
    psf_ism,
    lambda_value=FIXED_LAMBDA,
    weight_map=weight_map,
    max_iter=RBN_MAX_ITER,
    stop_tol=1e-6,
    stop_criteria="step_size",
    method="sgp",
    regularizer_norm="l2",
    initialization="flat",
    normalize=True,
    process=PROCESS,
    idx_in_focus=IN_FOCUS_PLANE,
    rec_to_save="last",
    step_size_search="line_search",
    line_search_params=(1e18, 2),
    verbose=False,
)

# rec_to_save="last" returns a single saved object.
fixed_obj = fixed_history[-1]

# Sum over K and select the in-focus plane.
fixed_volume = fixed_obj.sum(dim=-1).cpu().numpy()
fixed_rbn_image = fixed_volume[IN_FOCUS_PLANE]

print(f"Fixed-lambda reconstruction completed (lambda={FIXED_LAMBDA:g}).")

## 10. Compare the results

The panels show:

- the measured confocal image;
- conventional s²ISM;
- s²ISM-RbN with lambda selected by PRWP;
- s²ISM-RbN with the fixed lambda chosen in the settings cell.

For the two s²ISM-RbN reconstructions, the `K` reconstructed realizations are summed and only the in-focus plane is displayed.

In [ ]:
images = [
    confocal_image,
    s2ism_image,
    rbn_image,
    fixed_rbn_image,
]

titles = [
    "Raw confocal",
    f"s²ISM ({S2ISM_MAX_ITER} iterations)",
    f"s²ISM-RbN (PRWP, lambda={best_lambda:.3g})",
    f"s²ISM-RbN (fixed lambda={FIXED_LAMBDA:g})",
]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, image, title in zip(axes.ravel(), images, titles):
    gra.ShowImg(
        image,
        pixel_size_nm * 1e-3,
        clabel="Counts",
        fig=fig,
        ax=ax,
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 11. Optional: save the results

Uncomment the cell below to save the main outputs in a compact NumPy file.

In [ ]:
# OUTPUT_PATH = Path("results/s2ism_vs_s2ism_rbn.npz")
# OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#
# np.savez(
#     OUTPUT_PATH,
#     raw_confocal=confocal_image,
#     s2ism=s2ism_image,
#     s2ism_rbn_prwp=rbn_image,
#     s2ism_rbn_fixed=fixed_rbn_image,
#     best_lambda=best_lambda,
#     best_whiteness=best_whiteness,
#     fixed_lambda=FIXED_LAMBDA,
# )
#
# print(f"Saved to: {OUTPUT_PATH}")